In [ ]:

# ==========================================================
# EXPERIMENT NO. 6: REINFORCEMENT LEARNING
# Aim: Implement Reinforcement Learning using a Maze.
# Algorithm: Q-Learning
# ==========================================================


# STEP 1: Import required libraries

import numpy as np
import matplotlib.pyplot as plt
import random

# Set random seed for reproducible results
np.random.seed(42)
random.seed(42)


# STEP 2: Create the maze environment

# 0 = Open path
# 1 = Wall
# 2 = Goal

maze = np.array([
    [0, 0, 1, 0, 0],
    [1, 0, 1, 0, 1],
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 2]
])

# Define starting position of the agent
start = (0, 0)

# Define goal position
goal = (4, 4)

# Get number of rows and columns
rows, cols = maze.shape

# Display maze
print("Maze Environment:")
print(maze)


# STEP 3: Define actions

# 0 = Up
# 1 = Down
# 2 = Left
# 3 = Right

actions = {
    0: (-1, 0),
    1: (1, 0),
    2: (0, -1),
    3: (0, 1)
}

action_names = ["Up", "Down", "Left", "Right"]

# Total number of possible actions
num_actions = 4


# STEP 4: Define rewards

# Reward for reaching the goal
goal_reward = 100

# Penalty for hitting a wall
wall_penalty = -10

# Small penalty for every movement
step_penalty = -1


# STEP 5: Initialize Q-table

# Q-table stores the expected value of each action
# for every possible position in the maze.

Q = np.zeros((rows, cols, num_actions))


# STEP 6: Define the environment movement function

def take_action(position, action):

    # Get current row and column
    row, col = position

    # Get movement direction
    move_row, move_col = actions[action]

    # Calculate new position
    new_row = row + move_row
    new_col = col + move_col

    # Check whether the new position is outside the maze
    if (
        new_row < 0 or new_row >= rows or
        new_col < 0 or new_col >= cols
    ):
        return position, wall_penalty, False

    # Check whether the new position contains a wall
    if maze[new_row, new_col] == 1:
        return position, wall_penalty, False

    # Check whether the agent reached the goal
    if maze[new_row, new_col] == 2:
        return (new_row, new_col), goal_reward, True

    # Normal movement
    return (new_row, new_col), step_penalty, False


# STEP 7: Define Q-Learning parameters

learning_rate = 0.1       # Alpha: learning speed
discount_factor = 0.9     # Gamma: future reward importance
epsilon = 1.0             # Initial exploration probability
epsilon_decay = 0.995     # Reduce exploration over time
minimum_epsilon = 0.01

episodes = 5000

# Store rewards from each episode
episode_rewards = []


# STEP 8: Train the Q-Learning agent

for episode in range(episodes):

    # Reset agent to starting position
    state = start

    total_reward = 0

    # Limit steps to avoid infinite loops
    for step in range(100):

        # Choose a random action (exploration)
        if random.uniform(0, 1) < epsilon:
            action = random.randint(0, num_actions - 1)

        # Choose the best known action (exploitation)
        else:
            action = np.argmax(Q[state[0], state[1]])

        # Perform selected action
        next_state, reward, done = take_action(state, action)

        # Get current Q-value
        old_value = Q[state[0], state[1], action]

        # Get maximum future Q-value
        next_max = np.max(
            Q[next_state[0], next_state[1]]
        )

        # Update Q-value using Q-Learning formula
        new_value = old_value + learning_rate * (
            reward + discount_factor * next_max - old_value
        )

        # Store updated Q-value
        Q[state[0], state[1], action] = new_value

        # Update current state
        state = next_state

        # Add reward
        total_reward += reward

        # Stop if goal is reached
        if done:
            break

    # Store total reward of this episode
    episode_rewards.append(total_reward)

    # Reduce exploration gradually
    epsilon = max(
        minimum_epsilon,
        epsilon * epsilon_decay
    )


print("\nTraining completed successfully!")
print("Total episodes:", episodes)
print("Final exploration rate:", round(epsilon, 4))


# STEP 9: Display the learned Q-table

print("\nLearned Q-Table:")

for row in range(rows):
    for col in range(cols):
        if maze[row, col] != 1:
            print(
                f"Position ({row}, {col}):",
                np.round(Q[row, col], 2)
            )


# STEP 10: Find the learned path

def find_path():

    # Start from initial position
    state = start

    # Store the path
    path = [state]

    visited = {state}

    # Follow the best action from the Q-table
    for step in range(50):

        # Select action with highest Q-value
        action = np.argmax(Q[state[0], state[1]])

        # Move the agent
        next_state, reward, done = take_action(state, action)

        # Stop if agent does not move or repeats a position
        if next_state == state or next_state in visited:
            break

        # Update position and path
        state = next_state
        path.append(state)
        visited.add(state)

        # Stop when goal is reached
        if done:
            break

    return path


# Get final learned path
best_path = find_path()

print("\nLearned Path:")
print(best_path)

if best_path[-1] == goal:
    print("\nAgent successfully reached the goal!")
else:
    print("\nAgent did not reach the goal. Try increasing episodes.")


# STEP 11: Visualize the maze and learned path

# Create a copy of the maze for visualization
visual_maze = maze.astype(float)

# Mark the learned path with a different value
for position in best_path:
    if position != start and position != goal:
        visual_maze[position] = 0.5

# Display maze
plt.figure(figsize=(7, 7))

plt.imshow(visual_maze, cmap="YlGnBu")

# Add grid lines
plt.xticks(range(cols))
plt.yticks(range(rows))
plt.grid(True)

# Mark starting position
plt.scatter(
    start[1],
    start[0],
    color="green",
    s=200,
    label="Start"
)

# Mark goal position
plt.scatter(
    goal[1],
    goal[0],
    color="red",
    s=200,
    label="Goal"
)

# Draw the learned path
path_rows = [p[0] for p in best_path]
path_cols = [p[1] for p in best_path]

plt.plot(
    path_cols,
    path_rows,
    color="orange",
    marker="o",
    linewidth=3,
    label="Learned Path"
)

plt.title("Q-Learning Maze Navigation")
plt.xlabel("Column")
plt.ylabel("Row")
plt.legend()
plt.show()


# STEP 12: Plot learning performance

# Calculate average reward over groups of 100 episodes
average_rewards = [
    np.mean(episode_rewards[i:i + 100])
    for i in range(0, episodes, 100)
]

plt.figure(figsize=(9, 5))

plt.plot(average_rewards)

plt.title("Q-Learning Training Performance")
plt.xlabel("Training Groups (100 Episodes)")
plt.ylabel("Average Reward")

plt.grid(True)
plt.show()


# ==========================================================
# END OF EXPERIMENT NO. 6
# ==========================================================
